In [ ]:
from pathlib import Path
import json
import random
import importlib

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from sklearn.ensemble import RandomForestClassifier
from sklearn.utils.class_weight import compute_class_weight
import joblib

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

try:
    tf = importlib.import_module("tensorflow")
except ModuleNotFoundError as exc:
    raise ModuleNotFoundError(
        "TensorFlow is not installed. Run the package install cell first."
    ) from exc

keras = tf.keras
layers = tf.keras.layers
tf.random.set_seed(SEED)

KAGGLE_INPUT = Path("/kaggle/input")
PROJECT_ROOT = Path("/kaggle/working")
DATA_ROOT = PROJECT_ROOT / "data" / "external"
MODELS_DIR = PROJECT_ROOT / "models"

DATA_ROOT.mkdir(parents=True, exist_ok=True)
MODELS_DIR.mkdir(parents=True, exist_ok=True)

Kaggle input: /kaggle/input
Project root: /kaggle/working
Data root: /kaggle/working/data/external
Models dir: /kaggle/working/models


In [ ]:
import zipfile
from urllib.request import urlretrieve

def download_github_zip(repo_slug: str, branch: str, out_dir: Path) -> Path:
    out_dir.mkdir(parents=True, exist_ok=True)
    zip_path = out_dir / f"{repo_slug.replace('/', '_')}-{branch}.zip"
    extract_root = out_dir / f"{repo_slug.split('/')[-1]}-{branch}"

    if extract_root.exists():
        print(f"Using cached repo: {extract_root}")
        return extract_root

    url = f"https://codeload.github.com/{repo_slug}/zip/refs/heads/{branch}"
    print(f"Downloading {url}")
    urlretrieve(url, zip_path)

    with zipfile.ZipFile(zip_path, "r") as zf:
        zf.extractall(out_dir)

    return extract_root


import kagglehub

print("Downloading Kaggle football tackles dataset...")
kaggle_download_path = Path(kagglehub.dataset_download("zaikali/football-tackles"))
tackle_data_roots = [kaggle_download_path]
print(f"Tackles dataset path: {kaggle_download_path}")

# Auto-download GitHub datasets
print("Downloading StatsBomb open-data from GitHub...")
statsbomb_repo_root = download_github_zip("statsbomb/open-data", "master", DATA_ROOT)
statsbomb_dir = statsbomb_repo_root

print("Downloading Metrica sample-data from GitHub...")
metrica_repo_root = download_github_zip("metrica-sports/sample-data", "master", DATA_ROOT)
metrica_dir = metrica_repo_root


Mounting files to /kaggle/input/datasets/zaikali/football-tackles...
Tackles dataset path: /kaggle/input/datasets/zaikali/football-tackles
Dataset setup complete.
tackle_data_roots: [PosixPath('/kaggle/input/datasets/zaikali/football-tackles')]
statsbomb_dir: /kaggle/working/data/external/open-data-master
metrica_dir: /kaggle/working/data/external/sample-data-master


In [ ]:
IMG_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

def infer_label_from_path(p: Path):
    s = str(p).lower()
    if "foul" in s:
        return 1
    if "clean_tackle" in s or "tackle" in s:
        return 0
    return None

def collect_image_samples(root: Path):
    if root is None or not root.exists():
        return []

    samples = []
    for p in root.rglob("*"):
        if p.suffix.lower() not in IMG_EXTS:
            continue
        y = infer_label_from_path(p)
        if y is None:
            continue
        samples.append((str(p), y))
    return samples

image_samples = []
for root in tackle_data_roots:
    image_samples.extend(collect_image_samples(Path(root)))

image_samples = list(dict.fromkeys(image_samples))

print(f"Found {len(image_samples)} labeled image samples")


Found 3380 labeled image samples


In [ ]:
paths = [p for p, _ in image_samples]
labels = [y for _, y in image_samples]

x_train, x_val, y_train, y_val = train_test_split(
    paths, labels, test_size=0.2, random_state=SEED, stratify=labels
)

classes = np.array([0, 1])
cw = compute_class_weight(class_weight="balanced", classes=classes, y=np.array(y_train))
class_weights = {int(c): float(w) for c, w in zip(classes, cw)}
print(f"Class weights: {class_weights}")

IMG_SIZE = (224, 224)
BATCH_SIZE = 32

def decode_and_resize(path, label):
    img = tf.io.read_file(path)
    img = tf.image.decode_image(img, channels=3, expand_animations=False)
    img = tf.image.resize(img, IMG_SIZE)
    img = tf.cast(img, tf.float32) / 255.0
    return img, tf.cast(label, tf.float32)

train_ds = tf.data.Dataset.from_tensor_slices((x_train, y_train))
train_ds = train_ds.shuffle(len(x_train), seed=SEED).map(
    decode_and_resize, num_parallel_calls=tf.data.AUTOTUNE
).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

val_ds = tf.data.Dataset.from_tensor_slices((x_val, y_val))
val_ds = val_ds.map(decode_and_resize, num_parallel_calls=tf.data.AUTOTUNE).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

print(f"Train samples: {len(x_train)}, Val samples: {len(x_val)}")

I0000 00:00:1775926505.355389      55 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13757 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1775926505.362923      55 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13757 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


Train samples: 2704, Val samples: 676


In [ ]:
# Image model (foul vs clean tackle)
data_augmentation = keras.Sequential(
    [
        layers.RandomFlip("horizontal"),
        layers.RandomRotation(0.08),
        layers.RandomZoom(0.15),
        layers.RandomContrast(0.15),
        layers.RandomBrightness(0.12),
    ],
    name="data_augmentation",
)

base = keras.applications.MobileNetV2(include_top=False, input_shape=(224, 224, 3), weights="imagenet")
base.trainable = False

inputs = keras.Input(shape=(224, 224, 3))
x = data_augmentation(inputs, training=True)
x = keras.applications.mobilenet_v2.preprocess_input(x * 255.0)
x = base(x, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.35)(x)
outputs = layers.Dense(1, activation="sigmoid")(x)

image_model = keras.Model(inputs, outputs)
image_model.compile(
    optimizer=keras.optimizers.Adam(1e-3),
    loss="binary_crossentropy",
    metrics=["accuracy", keras.metrics.AUC(name="auc")],
)

callbacks = [
    keras.callbacks.EarlyStopping(monitor="val_auc", mode="max", patience=4, restore_best_weights=True)
]

history = image_model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=12,
    callbacks=callbacks,
    class_weight=class_weights,
    verbose=1,
)

9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
Epoch 1/12


I0000 00:00:1775926521.752483     134 service.cc:152] XLA service 0x7fbd040416e0 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1775926521.752528     134 service.cc:160]   StreamExecutor device (0): Tesla T4, Compute Capability 7.5
I0000 00:00:1775926521.752533     134 service.cc:160]   StreamExecutor device (1): Tesla T4, Compute Capability 7.5
I0000 00:00:1775926522.943428     134 cuda_dnn.cc:529] Loaded cuDNN version 91002
2026-04-11 16:55:31.945324: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-04-11 16:55:32.081855: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
I0000 00:00:1775926534.332756     134 device_co

84/85 ━━━━━━━━━━━━━━━━━━━━ 0s 64ms/step - accuracy: 0.5064 - auc: 0.5224 - loss: 0.7741

2026-04-11 16:55:48.862061: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-04-11 16:55:49.002479: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-04-11 16:55:49.137938: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.


85/85 ━━━━━━━━━━━━━━━━━━━━ 0s 201ms/step - accuracy: 0.5070 - auc: 0.5232 - loss: 0.7734

2026-04-11 16:56:05.876991: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-04-11 16:56:06.035061: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2026-04-11 16:56:06.170573: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.


85/85 ━━━━━━━━━━━━━━━━━━━━ 52s 401ms/step - accuracy: 0.5076 - auc: 0.5240 - loss: 0.7727 - val_accuracy: 0.6701 - val_auc: 0.7360 - val_loss: 0.6042
Epoch 2/12
85/85 ━━━━━━━━━━━━━━━━━━━━ 5s 59ms/step - accuracy: 0.6505 - auc: 0.7117 - loss: 0.6183 - val_accuracy: 0.7219 - val_auc: 0.8185 - val_loss: 0.5407
Epoch 3/12
85/85 ━━━━━━━━━━━━━━━━━━━━ 5s 60ms/step - accuracy: 0.7248 - auc: 0.8096 - loss: 0.5323 - val_accuracy: 0.7707 - val_auc: 0.8578 - val_loss: 0.5002
Epoch 4/12
85/85 ━━━━━━━━━━━━━━━━━━━━ 5s 62ms/step - accuracy: 0.7622 - auc: 0.8394 - loss: 0.5023 - val_accuracy: 0.7840 - val_auc: 0.8811 - val_loss: 0.4754
Epoch 5/12
85/85 ━━━━━━━━━━━━━━━━━━━━ 5s 59ms/step - accuracy: 0.7879 - auc: 0.8716 - loss: 0.4612 - val_accuracy: 0.7870 - val_auc: 0.8971 - val_loss: 0.4606
Epoch 6/12
85/85 ━━━━━━━━━━━━━━━━━━━━ 5s 59ms/step - accuracy: 0.8011 - auc: 0.8906 - loss: 0.4338 - val_accuracy: 0.8180 - val_auc: 0.9101 - val_loss: 0.4290
Epoch 7/12
85/85 ━━━━━━━━━━━━━━━━━━━━ 5s 59ms/step - ac

In [ ]:
from sklearn.metrics import f1_score

base.trainable = True
for layer in base.layers[:-40]:
    layer.trainable = False

image_model.compile(
    optimizer=keras.optimizers.Adam(1e-5),
    loss="binary_crossentropy",
    metrics=["accuracy", keras.metrics.AUC(name="auc")],
)
history_ft = image_model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=8,
    callbacks=callbacks,
    class_weight=class_weights,
    verbose=1,
)

image_model_path = MODELS_DIR / "tackle_foul_classifier.keras"
image_model.save(image_model_path)
print(f"Saved image model to: {image_model_path}")

# Threshold sweep on validation split and export best threshold
val_probs = image_model.predict(val_ds, verbose=0).ravel()
y_val_np = np.array(y_val, dtype=np.int32)

thresholds = np.linspace(0.10, 0.90, 81)
best = {"threshold": 0.5, "f1": -1.0}
for t in thresholds:
    preds = (val_probs >= t).astype(np.int32)
    f1 = float(f1_score(y_val_np, preds, zero_division=0))
    if f1 > best["f1"]:
        best = {"threshold": float(t), "f1": f1}

best_threshold = float(best["threshold"])
print(f"Best threshold by val F1: {best_threshold:.3f} (F1={best['f1']:.4f})")

threshold_cfg = {
    "foul_decision_threshold": best_threshold,
    "metric": "val_f1",
    "val_f1": float(best["f1"]),
    "n_val_samples": int(len(y_val_np)),
}
threshold_cfg_path = MODELS_DIR / "tackle_foul_config.json"
with open(threshold_cfg_path, "w", encoding="utf-8") as f:
    json.dump(threshold_cfg, f, indent=2)
print(f"Saved threshold config to: {threshold_cfg_path}")

Epoch 1/8
85/85 ━━━━━━━━━━━━━━━━━━━━ 41s 263ms/step - accuracy: 0.6798 - auc: 0.7725 - loss: 0.7179 - val_accuracy: 0.7500 - val_auc: 0.9276 - val_loss: 0.4770
Epoch 2/8
85/85 ━━━━━━━━━━━━━━━━━━━━ 5s 59ms/step - accuracy: 0.8071 - auc: 0.8839 - loss: 0.4324 - val_accuracy: 0.7737 - val_auc: 0.9330 - val_loss: 0.4542
Epoch 3/8
85/85 ━━━━━━━━━━━━━━━━━━━━ 5s 60ms/step - accuracy: 0.8866 - auc: 0.9535 - loss: 0.2881 - val_accuracy: 0.8107 - val_auc: 0.9413 - val_loss: 0.3997
Epoch 4/8
85/85 ━━━━━━━━━━━━━━━━━━━━ 5s 59ms/step - accuracy: 0.9095 - auc: 0.9670 - loss: 0.2480 - val_accuracy: 0.8388 - val_auc: 0.9502 - val_loss: 0.3521
Epoch 5/8
85/85 ━━━━━━━━━━━━━━━━━━━━ 5s 60ms/step - accuracy: 0.9300 - auc: 0.9807 - loss: 0.2049 - val_accuracy: 0.8476 - val_auc: 0.9575 - val_loss: 0.3258
Epoch 6/8
85/85 ━━━━━━━━━━━━━━━━━━━━ 5s 60ms/step - accuracy: 0.9609 - auc: 0.9914 - loss: 0.1513 - val_accuracy: 0.8802 - val_auc: 0.9668 - val_loss: 0.2748
Epoch 7/8
85/85 ━━━━━━━━━━━━━━━━━━━━ 5s 60ms/step 

In [11]:
# Build an auxiliary event model from StatsBomb events
# Label convention here:
# - 1: foul-related event
# - 0: duel/challenge without foul

def parse_statsbomb_events(events_root: Path, max_files=800):
    rows = []
    files = sorted(events_root.glob("*.json"))[:max_files]

    for fp in files:
        try:
            data = json.loads(fp.read_text(encoding="utf-8"))
        except Exception:
            continue

        for ev in data:
            t = (ev.get("type") or {}).get("name", "")
            if t not in {"Duel", "Foul Committed", "Foul Won"}:
                continue

            loc = ev.get("location") or [None, None]
            minute = ev.get("minute", 0)
            second = ev.get("second", 0)
            period = ev.get("period", 1)
            duration = ev.get("duration", 0.0) or 0.0

            y = 1 if "foul" in t.lower() else 0

            rows.append(
                {
                    "x": float(loc[0]) if loc[0] is not None else -1.0,
                    "y": float(loc[1]) if loc[1] is not None else -1.0,
                    "minute": float(minute),
                    "second": float(second),
                    "period": float(period),
                    "duration": float(duration),
                    "label": int(y),
                }
            )

    return pd.DataFrame(rows)

events_root = None
if statsbomb_dir is not None:
    statsbomb_dir = Path(statsbomb_dir)
    candidate_1 = statsbomb_dir / "data" / "events"
    candidate_2 = statsbomb_dir / "events"
    if candidate_1.exists():
        events_root = candidate_1
    elif candidate_2.exists():
        events_root = candidate_2

events_df = parse_statsbomb_events(events_root) if events_root is not None else pd.DataFrame()
print(f"StatsBomb event rows: {len(events_df)}")

if len(events_df) >= 200:
    X = events_df[["x", "y", "minute", "second", "period", "duration"]]
    y = events_df["label"]

    X_train, X_val, y_train, y_val = train_test_split(
        X, y, test_size=0.2, random_state=SEED, stratify=y
    )

    event_model = RandomForestClassifier(
        n_estimators=300, random_state=SEED, n_jobs=-1, class_weight="balanced"
    )
    event_model.fit(X_train, y_train)

    y_pred = event_model.predict(X_val)
    print(classification_report(y_val, y_pred, digits=3))

    event_model_path = MODELS_DIR / "tackle_event_classifier.joblib"
    joblib.dump(event_model, event_model_path)
    print(f"Saved event model to: {event_model_path}")
else:
    print("Not enough event rows to train auxiliary event model.")

StatsBomb event rows: 99096
              precision    recall  f1-score   support

           0      0.617     0.742     0.674     11735
           1      0.469     0.332     0.389      8085

    accuracy                          0.574     19820
   macro avg      0.543     0.537     0.531     19820
weighted avg      0.557     0.574     0.557     19820

Saved event model to: /kaggle/working/models/tackle_event_classifier.joblib
